# 🎬 Netflix Recommendation System — Personalized Content Discovery
**Cult Open Projects 2026 | AI/ML Track | IIT Roorkee**

This notebook builds a complete recommendation engine on the Netflix Prize Dataset using:
- **SVD** (Singular Value Decomposition)
- **ALS** (Alternating Least Squares)

**Mandatory Metrics:** RMSE + MAP@10

All outputs are saved to `/kaggle/working/` for the interactive dashboard.

## 1. Setup & Dependencies

In [ ]:
!pip install scikit-surprise implicit --quiet

import os, json, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('ggplot')
sns.set_palette('husl')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)
SEED = 42
np.random.seed(SEED)
print("Setup complete!")

## 2. Load Netflix Prize Data (Memory Efficient)

In [ ]:
DATA_DIR = '/kaggle/input/netflix-prize-data'

def load_netflix_data_fast(data_dir, num_files=2):
    """
    Load data efficiently to prevent Out-Of-Memory (OOM) errors on Kaggle.
    We load only the first `num_files` (out of 4) to stay well within memory limits.
    """
    files = sorted([f for f in os.listdir(data_dir) if f.startswith('combined_data') and f.endswith('.txt')])
    files = files[:num_files]
    print(f"Loading {len(files)} files to prevent memory issues...")
    
    dfs = []
    for fname in files:
        print(f"  → Processing {fname}...")
        filepath = os.path.join(data_dir, fname)
        
        # Read the file line by line to extract movie IDs and save intermediate csv
        tmp_csv = f"/kaggle/working/tmp_{fname}.csv"
        with open(filepath, 'r') as f_in, open(tmp_csv, 'w') as f_out:
            f_out.write("user_id,movie_id,rating,date\n")
            movie_id = -1
            for line in f_in:
                line = line.strip()
                if line.endswith(':'):
                    movie_id = line[:-1]
                else:
                    # Write: user_id, movie_id, rating, date
                    parts = line.split(',')
                    f_out.write(f"{parts[0]},{movie_id},{parts[1]},{parts[2]}\n")
        
        # Load the clean CSV into pandas efficiently
        df_chunk = pd.read_csv(tmp_csv, dtype={'user_id': np.int32, 'movie_id': np.int16, 'rating': np.float32})
        df_chunk['date'] = pd.to_datetime(df_chunk['date'])
        dfs.append(df_chunk)
        
        # Cleanup temp file
        os.remove(tmp_csv)
        
    df = pd.concat(dfs, ignore_index=True)
    print(f"\n✅ {len(df):,} ratings | {df['user_id'].nunique():,} users | {df['movie_id'].nunique():,} movies")
    return df

def load_movies(data_dir):
    movies = []
    with open(os.path.join(data_dir, 'movie_titles.csv'), 'r', encoding='latin-1') as f:
        for line in f:
            p = line.strip().split(',', 2)
            yr = p[1].strip() if p[1].strip() != 'NULL' else None
            movies.append((int(p[0]), yr, p[2].strip() if len(p)>2 else 'Unknown'))
    dm = pd.DataFrame(movies, columns=['movie_id','year','title'])
    dm['year'] = pd.to_numeric(dm['year'], errors='coerce')
    return dm

df_full = load_netflix_data_fast(DATA_DIR, num_files=2)
df_movies = load_movies(DATA_DIR)
print(f"Movies metadata: {len(df_movies):,} titles")

## 3. Smart Sampling (20% of active users)
We sample active users (≥20 ratings) to ensure collaborative filtering has enough data to learn patterns, while keeping compute time low.

In [ ]:
# We are using 100% of the active users for maximum competition accuracy!
# This will take significantly longer to train but yields the best possible RMSE.
SAMPLE_FRAC = 1.0
MIN_RATINGS = 20

uc = df_full['user_id'].value_counts()
active = uc[uc >= MIN_RATINGS].index
sampled = np.random.choice(active, size=int(len(active)*SAMPLE_FRAC), replace=False)
df = df_full[df_full['user_id'].isin(set(sampled))].copy().reset_index(drop=True)

print(f"Full Data: {len(df_full):,} ratings | {df_full['user_id'].nunique():,} users")
print(f"Sampled Data: {len(df):,} ratings | {df['user_id'].nunique():,} users | {df['movie_id'].nunique():,} movies")
del df_full; gc.collect()

## 4. Exploratory Data Analysis
**Task A: EDA & Business Implications**

In this section, we analyze user activity, content popularity, rating distributions, and data sparsity.

**Business & Technical Implications:**
1. **Data Sparsity:** Netflix data is incredibly sparse (typically >98%). Technically, this means traditional User-User collaborative filtering will struggle because finding overlapping users is rare. This justifies our use of **Latent Factor Models (SVD/ALS)** which reduce dimensionality to find hidden patterns.
2. **Content Popularity (Long-Tail):** A small percentage of blockbuster movies receives the vast majority of ratings. From a business perspective, the recommendation engine must balance suggesting popular hits (safe bets) with discovering niche content (long-tail) to keep users engaged with the platform's deeper catalog.
3. **User Activity:** Some power users rate thousands of movies, while most rate very few. The system must be robust to "cold-start" or "low-activity" users.
4. **Rating Distribution:** Users generally rate movies they like (left-skewed distribution, average rating ~3.6). We must adjust for this optimism bias.

In [ ]:
n_users = df['user_id'].nunique()
n_movies = df['movie_id'].nunique()
n_ratings = len(df)
sparsity = 1 - n_ratings / (n_users * n_movies)

print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(f"  Ratings:   {n_ratings:,}")
print(f"  Users:     {n_users:,}")
print(f"  Movies:    {n_movies:,}")
print(f"  Sparsity:  {sparsity*100:.2f}%")
print(f"  Mean:      {df['rating'].mean():.3f}")
print(f"  Median:    {df['rating'].median():.1f}")
print(f"  Date:      {df['date'].min().date()} → {df['date'].max().date()}")

# Rating distribution data
rating_counts = df['rating'].value_counts().sort_index()
rating_dist = {str(int(k)): int(v) for k, v in rating_counts.items()}

# User activity data
rpu = df.groupby('user_id').size()
user_activity = {
    'mean': float(rpu.mean()), 'median': float(rpu.median()),
    'min': int(rpu.min()), 'max': int(rpu.max()), 'std': float(rpu.std()),
    'percentiles': {str(p): float(rpu.quantile(p/100)) for p in [10,25,50,75,90,95,99]}
}

# Temporal trends
monthly = df.set_index('date').resample('M').size()
temporal = {d.strftime('%Y-%m'): int(v) for d, v in monthly.items()}

# Movie popularity
pop = df.groupby('movie_id').agg(count=('rating','size'), avg_rating=('rating','mean')).reset_index()
pop = pop.merge(df_movies[['movie_id','title','year']], on='movie_id', how='left')
top50 = pop.nlargest(50, 'count')
movie_pop = top50[['movie_id','title','year','count','avg_rating']].to_dict('records')

# Save EDA stats
eda_stats = {
    'n_users': n_users, 'n_movies': n_movies, 'n_ratings': n_ratings,
    'sparsity': round(sparsity, 6), 'mean_rating': round(df['rating'].mean(), 4),
    'median_rating': float(df['rating'].median()),
    'rating_distribution': rating_dist, 'user_activity': user_activity,
    'temporal_trends': temporal, 'movie_popularity': movie_pop
}
with open(os.path.join(OUTPUT_DIR, 'eda_stats.json'), 'w') as f:
    json.dump(eda_stats, f, indent=2, default=str)
print(f"\n✅ EDA stats saved → eda_stats.json")

In [ ]:
# --- EDA Plots ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Netflix Prize — Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Rating distribution
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
axes[0,0].bar(rating_counts.index, rating_counts.values, color=colors, edgecolor='white')
axes[0,0].set_xlabel('Rating'); axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Rating Distribution', fontweight='bold')
for r, c in rating_counts.items():
    axes[0,0].text(r, c+c*0.02, f'{c:,}\n({c/n_ratings*100:.1f}%)', ha='center', fontsize=8)

# User activity
axes[0,1].hist(rpu.values, bins=100, color='#3498db', alpha=0.8, edgecolor='white')
axes[0,1].axvline(rpu.median(), color='red', ls='--', label=f'Median: {rpu.median():.0f}')
axes[0,1].axvline(rpu.mean(), color='orange', ls='--', label=f'Mean: {rpu.mean():.0f}')
axes[0,1].set_xlabel('Ratings per User'); axes[0,1].set_ylabel('Users')
axes[0,1].set_title('User Activity (log scale)', fontweight='bold')
axes[0,1].set_xscale('log'); axes[0,1].legend()

# Temporal trends
axes[1,0].fill_between(monthly.index, monthly.values, alpha=0.3, color='#2ecc71')
axes[1,0].plot(monthly.index, monthly.values, color='#27ae60', lw=2)
axes[1,0].set_xlabel('Date'); axes[1,0].set_ylabel('Ratings/Month')
axes[1,0].set_title('Rating Volume Over Time', fontweight='bold')

# Top 20 movies
t20 = top50.head(20)
axes[1,1].barh(range(20), t20['count'], color='#9b59b6', alpha=0.8)
axes[1,1].set_yticks(range(20))
axes[1,1].set_yticklabels(t20['title'].fillna('?').str[:35], fontsize=8)
axes[1,1].set_xlabel('Ratings'); axes[1,1].set_title('Top 20 Movies', fontweight='bold')
axes[1,1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_overview.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved → eda_overview.png")

## 5. Train / Test Split Methodology
**Task E: Evaluation Setup**

**Methodology:** We use a random 80/20 train-test split. 80% of the ratings are used to train the models, and 20% are held out to test prediction accuracy and ranking quality. 
Because we are evaluating MAP@10, the test set contains actual movies the user interacted with. We define a **"relevant" movie as having a rating ≥ 3.5**, exactly as specified in the problem statement.

In [ ]:
mask = np.random.rand(len(df)) < 0.8
train_df = df[mask].copy().reset_index(drop=True)
test_df = df[~mask].copy().reset_index(drop=True)
print(f"Train: {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")

## 6. Model 1 — SVD (Matrix Factorization)
**Task B: Recommendation Model Development**

**Methodology:** Singular Value Decomposition (SVD), popularized by Simon Funk during the Netflix Prize, factorizes the user-item rating matrix into two lower-dimensional matrices representing latent user preferences and latent item features.

**Suitability:** It is highly suitable for explicitly rated, highly sparse datasets like Netflix because it learns hidden features (e.g., "how much comedy is in this movie" vs "how much does this user like comedy") and naturally handles missing data by predicting the dot product of these latent vectors plus global/user/item biases.

In [ ]:
from surprise import SVD, Dataset, Reader, accuracy

print("Training SVD (Heavy Configuration)...")
reader = Reader(rating_scale=(1, 5))
train_data = Dataset.load_from_df(train_df[['user_id','movie_id','rating']], reader)
trainset = train_data.build_full_trainset()

# Increased factors to 200 and epochs to 50 for deep learning extraction
svd = SVD(n_factors=200, n_epochs=50, lr_all=0.005, reg_all=0.02, random_state=SEED)
svd.fit(trainset)

testset = list(zip(test_df['user_id'], test_df['movie_id'], test_df['rating']))
svd_preds = svd.test(testset)
svd_rmse = accuracy.rmse(svd_preds)
print(f"✅ SVD RMSE: {svd_rmse:.4f}")

## 7. Model 2 — ALS (Alternating Least Squares)
**Task C: Model Comparison (Matrix Factorization vs Implicit Collaborative Filtering)**

**Methodology:** ALS optimizes the matrix factorization problem by holding item factors constant to solve for user factors, and then holding user factors constant to solve for item factors, alternating back and forth.

**Suitability:** ALS scales exceptionally well to massive datasets (like 100M+ Netflix ratings) and can be easily parallelized on CPUs/GPUs. Unlike SVD which focuses purely on explicit rating prediction error, ALS is highly effective at ranking items.

In [ ]:
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

print("Training ALS...")
users_arr = train_df['user_id'].unique()
items_arr = train_df['movie_id'].unique()
u_map = {u: i for i, u in enumerate(users_arr)}
i_map = {m: i for i, m in enumerate(items_arr)}
u_inv = {v: k for k, v in u_map.items()}
i_inv = {v: k for k, v in i_map.items()}

# In implicit >= 0.6.0, fit() expects a user_item matrix.
rows = train_df['user_id'].map(u_map).values
cols = train_df['movie_id'].map(i_map).values
vals = train_df['rating'].values.astype(np.float32)
sparse_mat = csr_matrix((vals, (rows, cols)), shape=(len(users_arr), len(items_arr)))

# Increased factors to 200 and iterations to 40 for max precision
als = AlternatingLeastSquares(factors=200, regularization=0.01, iterations=40, random_state=SEED)
als.fit(sparse_mat)

# Get user/item factor arrays (compatible with both old and new implicit versions)
try:
    _uf = als.user_factors
    _if = als.item_factors
    if hasattr(_uf, 'to_numpy'):
        _uf = _uf.to_numpy()
        _if = _if.to_numpy()
except Exception:
    _uf = np.array(als.user_factors)
    _if = np.array(als.item_factors)

# Vectorized ALS prediction (fast — avoids slow iterrows loop)
print("Predicting on test set (vectorized)...")
test_uids = test_df['user_id'].map(u_map)
test_mids = test_df['movie_id'].map(i_map)
known_mask = test_uids.notna() & test_mids.notna()

als_est = np.full(len(test_df), 3.0)  # default for unknown
known_idx = np.where(known_mask)[0]
u_indices = test_uids.iloc[known_idx].astype(int).values
m_indices = test_mids.iloc[known_idx].astype(int).values
# Batch dot product
als_est[known_idx] = np.clip(
    np.sum(_uf[u_indices] * _if[m_indices], axis=1), 1, 5
)

als_preds = list(zip(
    test_df['user_id'].values, test_df['movie_id'].values,
    test_df['rating'].values, als_est, [{}]*len(test_df)
))
als_rmse = np.sqrt(np.mean((test_df['rating'].values - als_est)**2))
print(f"ALS RMSE: {als_rmse:.4f}")

## 8. Evaluation — RMSE & MAP@10
**Task E: Evaluation Metrics**

We evaluate using two mandatory metrics:
1. **RMSE:** Measures rating prediction accuracy. It penalizes large errors heavily.
2. **MAP@10:** Measures recommendation ranking quality. 

**MAP@10 Procedure:**
1. We generate Top-10 predictions for each user in the test set.
2. We filter the test set for "relevant" movies (actual rating $\ge 3.5$).
3. For each user, we calculate Precision@k for $k=1..10$.
4. We average the precision scores at ranks where a relevant movie was recommended to get the Average Precision (AP).
5. We take the Mean of AP across all users to get MAP@10.

In [ ]:
def calc_map(preds, k=10, thr=3.5):
    ue = defaultdict(list)
    for p in preds:
        if hasattr(p, 'uid'): ue[p.uid].append((p.est, p.r_ui))
        else: ue[p[0]].append((p[3], p[2]))
    aps = []
    for uid, rats in ue.items():
        rats.sort(key=lambda x: x[0], reverse=True)
        tk = rats[:k]; nr = 0; ap = 0.0
        for i, (e, t) in enumerate(tk):
            if t >= thr: nr += 1; ap += nr/(i+1)
        aps.append(ap/nr if nr > 0 else 0.0)
    return np.mean(aps)

def calc_prec_rec(preds, k=10, thr=3.5):
    ue = defaultdict(list)
    for p in preds:
        if hasattr(p, 'uid'): ue[p.uid].append((p.est, p.r_ui))
        else: ue[p[0]].append((p[3], p[2]))
    precs, recs = [], []
    for uid, rats in ue.items():
        tr = sum(1 for _,t in rats if t>=thr)
        if tr == 0: continue
        rats.sort(key=lambda x: x[0], reverse=True)
        nr = sum(1 for _,t in rats[:k] if t>=thr)
        precs.append(nr/k); recs.append(nr/tr)
    return np.mean(precs), np.mean(recs)

def calc_ndcg(preds, k=10):
    ue = defaultdict(list)
    for p in preds:
        if hasattr(p, 'uid'): ue[p.uid].append((p.est, p.r_ui))
        else: ue[p[0]].append((p[3], p[2]))
    ndcgs = []
    for uid, rats in ue.items():
        rats.sort(key=lambda x: x[0], reverse=True)
        dcg = sum(t/np.log2(i+2) for i,(_, t) in enumerate(rats[:k]))
        ideal = sorted(rats, key=lambda x: x[1], reverse=True)[:k]
        idcg = sum(t/np.log2(i+2) for i,(_, t) in enumerate(ideal))
        if idcg > 0: ndcgs.append(dcg/idcg)
    return np.mean(ndcgs)

svd_map = calc_map(svd_preds)
als_map = calc_map(als_preds)
svd_p, svd_r = calc_prec_rec(svd_preds)
als_p, als_r = calc_prec_rec(als_preds)
svd_ndcg = calc_ndcg(svd_preds)
als_ndcg = calc_ndcg(als_preds)
svd_mae = np.mean([abs(p.r_ui - p.est) for p in svd_preds])
als_mae = np.mean([abs(p[2]-p[3]) for p in als_preds])

eval_results = {
    'svd': {'rmse': round(svd_rmse,4), 'mae': round(svd_mae,4), 'map10': round(svd_map,4),
            'precision10': round(svd_p,4), 'recall10': round(svd_r,4), 'ndcg10': round(svd_ndcg,4)},
    'als': {'rmse': round(als_rmse,4), 'mae': round(als_mae,4), 'map10': round(als_map,4),
            'precision10': round(als_p,4), 'recall10': round(als_r,4), 'ndcg10': round(als_ndcg,4)}
}

print("\n" + "="*60)
print(f"{'Metric':<18} {'SVD':<12} {'ALS':<12}")
print("-"*42)
for m in ['rmse','mae','map10','precision10','recall10','ndcg10']:
    print(f"  {m:<16} {eval_results['svd'][m]:<12.4f} {eval_results['als'][m]:<12.4f}")
print("="*60)

with open(os.path.join(OUTPUT_DIR, 'evaluation_results.json'), 'w') as f:
    json.dump(eval_results, f, indent=2)
print("✅ Saved → evaluation_results.json")

In [ ]:
# Model Comparison Chart
metrics = ['RMSE ↓','MAE ↓','MAP@10 ↑','P@10 ↑','R@10 ↑','NDCG@10 ↑']
svd_v = [svd_rmse, svd_mae, svd_map, svd_p, svd_r, svd_ndcg]
als_v = [als_rmse, als_mae, als_map, als_p, als_r, als_ndcg]

fig, axes = plt.subplots(1, 6, figsize=(22, 5))
fig.suptitle('SVD vs ALS — Model Comparison', fontsize=15, fontweight='bold')
for i, (m, s, a) in enumerate(zip(metrics, svd_v, als_v)):
    bars = axes[i].bar(['SVD','ALS'], [s,a], color=['#3498db','#e74c3c'], edgecolor='white')
    axes[i].set_title(m, fontweight='bold', fontsize=10)
    for b, v in zip(bars, [s, a]):
        axes[i].text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved → model_comparison.png")

## 9. Model Comparison & Discussion
**Task C: Discussion of Approaches**

1. **Recommendation Quality:** SVD generally achieves lower RMSE because it explicitly minimizes prediction error. ALS often provides slightly worse RMSE but is structurally better at generating candidate rankings for implicit/top-k tasks.
2. **Training Complexity:** SVD uses Stochastic Gradient Descent (SGD) which updates sequentially. ALS solves least squares iteratively.
3. **Computational Efficiency:** ALS is drastically faster on large scales because the alternating steps can be parallelized perfectly across CPU cores or a GPU. SGD (SVD) is harder to parallelize.
4. **Practical Usability:** In a real business setting, ALS is often preferred for generating the "Top-10 rows" for a user's homepage because it scales to millions of users instantly and focuses on ranking, whereas SVD is excellent for predicting the exact star rating of a movie a user is currently looking at.

## 10. Generating Top-K Recommendations
**Task D: Recommendation Generation**

We now generate the Top-10 unseen movies for sampled users using both models.

In [ ]:
all_mids = df['movie_id'].unique()
sample_uids = train_df['user_id'].value_counts().head(20).index.tolist()

recs_output = {}
for uid in tqdm(sample_uids, desc="Generating recs"):
    rated = set(train_df[train_df['user_id']==uid]['movie_id'])
    cands = [m for m in all_mids if m not in rated]
    
    # SVD recs
    sp = [(m, svd.predict(uid, m).est) for m in cands[:3000]]
    sp.sort(key=lambda x: x[1], reverse=True)
    svd_top = sp[:10]
    
    # ALS recs
    if uid in u_map:
        u_idx = u_map[uid]
        try:
            # implicit >= 0.7.0 recommend API
            u_items = sparse_mat[u_idx] # user ratings row (1, n_items)
            ids, scores = als.recommend(u_idx, u_items, N=10, filter_already_liked_items=True)
        except TypeError:
            ids, scores = als.recommend(u_idx, sparse_mat.tocsr(), N=10, filter_already_liked_items=True)
        als_top = [(int(i_inv.get(int(ix), 0)), float(sc)) for ix, sc in zip(ids, scores)]
    else:
        als_top = []

    user_liked = train_df[train_df['user_id']==uid].sort_values('rating', ascending=False).head(5)
    user_liked = user_liked.merge(df_movies[['movie_id','title']], on='movie_id', how='left')

    recs_output[str(uid)] = {
        'user_id': int(uid),
        'n_rated': len(rated),
        'top_liked': [{'movie_id': int(r['movie_id']), 'title': r.get('title','?'), 'rating': float(r['rating'])}
                      for _, r in user_liked.iterrows()],
        'svd_recs': [{'rank': i+1, 'movie_id': int(m),
                      'title': df_movies[df_movies['movie_id']==m]['title'].values[0] if len(df_movies[df_movies['movie_id']==m])>0 else '?',
                      'predicted_score': round(s, 3)}
                     for i, (m, s) in enumerate(svd_top)],
        'als_recs': [{'rank': i+1, 'movie_id': int(m),
                      'title': df_movies[df_movies['movie_id']==m]['title'].values[0] if len(df_movies[df_movies['movie_id']==m])>0 else '?',
                      'predicted_score': round(s, 3)}
                     for i, (m, s) in enumerate(als_top)]
    }

with open(os.path.join(OUTPUT_DIR, 'recommendations.json'), 'w') as f:
    json.dump(recs_output, f, indent=2, default=str)
print(f"✅ Saved recs for {len(recs_output)} users → recommendations.json")

## 11. Success & Failure Analysis
**Task D: Analysis of Generated Recommendations**

Here we analyze the real-world performance of our generated Top-K recommendations.
- **Success Cases:** The model recommended a movie the user hadn't seen, and in our hidden test set, the user actually rated it $\ge 3.5$.
- **Failure Cases:** The model highly recommended a movie, but the user actually rated it $< 3.5$ in reality.

**Key Observations:** SVD and ALS successfully identify blockbuster hits for general users, but failures typically occur on highly polarizing movies (e.g., cult classics) where the latent factors misjudge a specific user's tolerance for niche genres.

In [ ]:
successes, failures = [], []
for uid_str, r in recs_output.items():
    uid = r['user_id']
    ut = test_df[test_df['user_id']==uid]
    for rec in r['svd_recs']:
        actual = ut[ut['movie_id']==rec['movie_id']]
        if len(actual) > 0:
            tr = float(actual['rating'].values[0])
            entry = {'user_id': uid, 'movie': rec['title'], 'predicted': rec['predicted_score'], 'actual': tr}
            (successes if tr >= 3.5 else failures).append(entry)

print("✅ SUCCESS CASES (predicted high, user liked):")
if successes:
    print(pd.DataFrame(successes[:10]).to_string(index=False))
print(f"\n❌ FAILURE CASES (predicted high, user didn't like):")
if failures:
    print(pd.DataFrame(failures[:10]).to_string(index=False))
total = len(successes) + len(failures)
if total > 0:
    print(f"\n📊 Hit accuracy: {len(successes)}/{total} = {len(successes)/total:.1%}")

sf_data = {'successes': successes[:50], 'failures': failures[:50],
           'accuracy': round(len(successes)/total, 4) if total > 0 else 0}
with open(os.path.join(OUTPUT_DIR, 'success_failure.json'), 'w') as f:
    json.dump(sf_data, f, indent=2, default=str)
print("✅ Saved → success_failure.json")

## 11. Explainable Recommendations

In [ ]:
explanations = []
for uid_str in list(recs_output.keys())[:10]:
    r = recs_output[uid_str]
    if not r['svd_recs']: continue
    rec = r['svd_recs'][0]
    liked = r['top_liked'][:3]
    exp = {
        'user_id': r['user_id'],
        'recommended': rec['title'],
        'predicted_score': rec['predicted_score'],
        'because_you_liked': [{'title': l['title'], 'rating': l['rating']} for l in liked],
        'explanation': f"We recommend '{rec['title']}' because you enjoyed " +
                       ', '.join(f"'{l['title']}' ({l['rating']}/5)" for l in liked) +
                       ". Users with similar taste also highly rated this movie."
    }
    explanations.append(exp)

with open(os.path.join(OUTPUT_DIR, 'explanations.json'), 'w') as f:
    json.dump(explanations, f, indent=2)
print("✅ Saved → explanations.json")

## 12. Save Movie Metadata for Dashboard

In [ ]:
df_movies.to_csv(os.path.join(OUTPUT_DIR, 'movie_metadata.csv'), index=False)
user_summary = train_df.groupby('user_id').agg(
    n_ratings=('rating','size'), avg_rating=('rating','mean'),
    min_rating=('rating','min'), max_rating=('rating','max')
).reset_index().head(500)
user_summary.to_csv(os.path.join(OUTPUT_DIR, 'user_summary.csv'), index=False)
print("✅ Metadata saved.")

## 13. Final Summary & Output Manifest

In [ ]:
output_files = os.listdir(OUTPUT_DIR)
print("\n" + "="*60)
print("  📦 OUTPUT FILES (for Dashboard)")
print("="*60)
for f in sorted(output_files):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:<30} {size/1024:.1f} KB")

print(f"\n{'='*60}")
print("  🎯 Download ALL files from /kaggle/working/ for the dashboard!")
print(f"{'='*60}")